In [12]:
pip install cudf-polars-cu11==25.06

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached polars-1.28.1-cp39-abi3-win_amd64.whl.metadata (15 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [72 lines of output]
      INFO:wheel-stub:Testing wheel pylibcudf_cu11-25.6.0-cp310-cp310-manylinux_2_24_aarch64.manylinux_2_28_aarch64.whl against tag cp310-cp310-manylinux_2_28_aarch64
      INFO:wheel-stub:Testing wheel pylibcudf_cu11-25.6.0-cp310-cp310-manylinux_2_24_aarch64.manylinux_2_28_aarch64.whl against tag cp310-cp310-manylinux_2_24_aarch64
      INFO:wheel-stub:Testing wheel pylibcudf_cu11-25.6.0-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl against tag cp310-cp310-manylinux_2_28_x86_64
      INFO:wheel-stub:Testing wheel pylibcudf_cu11-25.6.0-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl against tag cp310-cp310-manylinux_2_24_x86_64
      INFO:wheel-stub:Testing wheel pylibcudf_cu11-25.6.0-cp311-cp311-manylinux_2_24_aarch64.manylinux_2_28_aarch64.whl against tag cp311-cp311-manylinux_2_28_aarch64
      INFO:wheel-stu

In [1]:
import polars as pl

In [2]:
# GPU-движок: Polars сам выберет GPU, при ошибке — откатится на CPU
gpu_engine = pl.GPUEngine(
    device=0,          # индекс GPU (0 = первая видеокарта)
    raise_on_fail=False  # False = тихий откат на CPU если GPU не поддерживает операцию
)


In [4]:
# Читаем parquet через LazyFrame (ленивые вычисления — основа GPU-режима)
bus = pl.scan_parquet('business_cards_MDQ.parquet')
con = pl.scan_parquet('consumer_cards_MDQ.parquet')
mer = pl.scan_parquet('merchants_reference.parquet')

In [5]:
# Добавляем target
bus = bus.with_columns(pl.lit(1).alias('target'))
con = con.with_columns(pl.lit(0).alias('target'))

In [7]:
# Конкатенация
df = pl.concat([bus, con])

In [8]:
# Merge с merchants
df = df.join(mer, on='merchant_id', how='left')

In [10]:
# Парсим datetime и генерируем признаки
df = df.with_columns([
    pl.col('transaction_timestamp').cast(pl.Datetime).alias('dt')
]).with_columns([
    pl.col('dt').dt.hour().alias('hour'),
    pl.col('dt').dt.weekday().alias('dayofweek'),  # 1=пн, 7=вс в Polars
]).with_columns([
    pl.col('dayofweek').is_in([6, 7]).cast(pl.Int32).alias('is_weekend'),
    pl.col('hour').is_between(9, 18).cast(pl.Int32).alias('is_business_hour'),
])

In [13]:
df